# **README**


# MODEL CITATION


# DATASET CITATIONS

## KVASIR V2 Dataset
/kaggle/input/datasets/yasserhessein/the-kvasir-dataset

# CONTACT
For further assitance or queries, you may contact the author via 

# ORIGINAL CODES


# GRAPHICAL ABSTRACT
main image.png](<attachment:journal-2 main image.png>)

# LOAD DEPENDENCIES

In [1]:
# #Installs the library to measure the FLOPs of the model. 
# #Don't mind the error received, after installation.

!pip install keras-flops

In [2]:
# ==========================================
# LOAD ALL REQUIRED DEPENDENCIES
# ==========================================

import os
import cv2
import time
import pickle
import logging
import itertools
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

# Scikit-learn
from sklearn.utils import class_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

# ==========================================
# REQUIRED IMPORTS FOR FUSION MODEL
# ==========================================
# TensorFlow / Keras Core
from tensorflow.keras.models import Model, load_model
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    AveragePooling2D,
    GlobalAveragePooling2D,
    BatchNormalization,
    Dense,
    Add,
    AlphaDropout
)

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# Backbone Models for Fusion
from tensorflow.keras.applications.efficientnet import EfficientNetB0 as trainable_model_a
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2 as trainable_model_b
from tensorflow.keras.applications.resnet_v2 import ResNet50V2 as trainable_model_c

# Suppress TensorFlow warnings
tf.get_logger().setLevel(logging.ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print("Packages successfully imported!")

# ==========================================
# GPU CHECK
# ==========================================

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("GPU Acceleration Enabled:", len(gpus) > 0)

2026-03-08 08:15:09.734119: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772957710.148479      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772957710.237970      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772957711.174623      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772957711.174683      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772957711.174686      55 computation_placer.cc:177] computation placer alr

Packages successfully imported!
GPU Acceleration Enabled: True


# Feature Attention Block

In [ ]:
# ==========================================
# Feature Attention Block
# ==========================================
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Reshape, Multiply, Lambda
import tensorflow.keras.backend as K

def se_block(x, reduction=16):
    ch = x.shape[-1]
    se = GlobalAveragePooling2D()(x)
    se = Dense(ch//reduction, activation='relu')(se)
    se = Dense(ch, activation='sigmoid')(se)
    se = Reshape((1,1,ch))(se)
    return Multiply()([x,se])

def self_attention_block(x):
    ch = x.shape[-1]
    f = Conv2D(ch//8,1,padding='same')(x)
    g = Conv2D(ch//8,1,padding='same')(x)
    h = Conv2D(ch,1,padding='same')(x)

    s = Lambda(lambda z: K.batch_dot(
        K.reshape(z[0],(-1,z[0].shape[1]*z[0].shape[2],z[0].shape[3])),
        K.reshape(z[1],(-1,z[1].shape[1]*z[1].shape[2],z[1].shape[3])),
        axes=[2,2]))([f,g])

    beta = Activation('softmax')(s)

    o = Lambda(lambda z: K.batch_dot(
        z[0],
        K.reshape(z[1],(-1,z[1].shape[1]*z[1].shape[2],z[1].shape[3]))
    ))([beta,h])

    o = Lambda(lambda z: K.reshape(z,(-1,x.shape[1],x.shape[2],ch)))(o)

    return Add()([o,x])

def feature_attention_block(x):
    x = se_block(x)
    x = self_attention_block(x)
    return x


# SET CONSTANTS

In [ ]:
# ==========================================
# SET TRAINING CONSTANTS (OPTIMIZED)
# ==========================================

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

batch_size = 32
epochs = 60
architecture = 'HAFusionNet'

optimizer = Adam(learning_rate=0.00003)

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=3,
    verbose=1,
    mode='max',
    min_lr=1e-6
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

callbacks = [reduce_lr, early_stop]

I0000 00:00:1772957744.435617      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772957744.438360      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:
#Set model identifiers
DCNN_A = 'DCNN_A' #EfficientNetB0
DCNN_B = 'DCNN_B' #MobileNetV2
DCNN_C = 'DCNN_C' #ResNet50V2


# CUSTOM FUNCTIONS

In [6]:
#Custom save function
def save_h(file, history):
    with open(file + '/' + architecture + '/' + architecture + '.history', 'wb') as file_pi:
        pickle.dump(history, file_pi)
    print("history saved")

# DATASET PREPARATION

In [7]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [8]:
print(os.listdir("/kaggle/input/datasets/yasserhessein/the-kvasir-dataset/kvasir-dataset-v2"))

['dyed-lifted-polyps', 'normal-z-line', 'dyed-resection-margins', 'normal-pylorus', 'normal-cecum', 'polyps', 'ulcerative-colitis', 'esophagitis']


In [9]:
#LOAD THE DATA
main_dir = '../kaggle/input/datasets/yasserhessein/the-kvasir-dataset'

#Data split
train_data_dir = main_dir + "train/"
validation_data_dir = main_dir + "val/"
test_data_dir = main_dir + "test/"

#Set image shapes
img_rows, img_cols = 224, 224
input_shape = (img_rows,img_cols,3)
model_input = Input(shape=input_shape)

#Prompt
print("Data folders found!")
print("The Input size is set to ", model_input) 

Data folders found!
The Input size is set to  <KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor>


In [10]:
import os
import shutil
from pathlib import Path

src = Path("/kaggle/input/datasets/yasserhessein/the-kvasir-dataset")
dst = Path("/kaggle/working/kvasir_fixed")

classes = [
    "dyed-lifted-polyps",
    "dyed-resection-margins",
    "esophagitis",
    "normal-cecum",
    "normal-pylorus",
    "normal-z-line",
    "polyps",
    "ulcerative-colitis"
]

for c in classes:
    (dst/c).mkdir(parents=True, exist_ok=True)

    image_dir = src/c/"images"

    for img in image_dir.glob("*.jpg"):
        shutil.copy(img, dst/c/img.name)

print("Dataset structure fixed!")

Dataset structure fixed!


In [11]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

# Original dataset path
dataset_path = Path("/kaggle/input/datasets/yasserhessein/the-kvasir-dataset/kvasir-dataset-v2")

# Output folder
output_path = Path("/kaggle/working/kvasir_combined")

train_dir = output_path / "train"
val_dir = output_path / "val"
test_dir = output_path / "test"

# Create folders
for folder in [train_dir, val_dir, test_dir]:
    folder.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Process each class
for class_name in os.listdir(dataset_path):

    class_path = dataset_path / class_name
    
    if not class_path.is_dir():
        continue

    images = list(class_path.glob("*.jpg"))

    print(class_name, ":", len(images))

    if len(images) == 0:
        continue

    train_imgs, temp_imgs = train_test_split(
        images,
        test_size=(VAL_RATIO + TEST_RATIO),
        random_state=42
    )

    val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=TEST_RATIO/(VAL_RATIO+TEST_RATIO),
        random_state=42
    )

    # Create class folders
    (train_dir / class_name).mkdir(parents=True, exist_ok=True)
    (val_dir / class_name).mkdir(parents=True, exist_ok=True)
    (test_dir / class_name).mkdir(parents=True, exist_ok=True)

    # Copy files
    for img in train_imgs:
        shutil.copy(img, train_dir / class_name / img.name)

    for img in val_imgs:
        shutil.copy(img, val_dir / class_name / img.name)

    for img in test_imgs:
        shutil.copy(img, test_dir / class_name / img.name)

print("Dataset split completed!")

dyed-lifted-polyps : 1000
normal-z-line : 1000
dyed-resection-margins : 1000
normal-pylorus : 1000
normal-cecum : 1000
polyps : 1000
ulcerative-colitis : 1000
esophagitis : 1000
Dataset split completed!


In [12]:
print(os.listdir("/kaggle/working/kvasir_combined"))

['train', 'val', 'test']


# DATA GENERATION & AUGMENTATION

In [13]:
# ==========================================
# DATA GENERATORS FOR KVASIR DATASET
# ==========================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_data_dir = "/kaggle/working/kvasir_combined/train"
validation_data_dir = "/kaggle/working/kvasir_combined/val"
test_data_dir = "/kaggle/working/kvasir_combined/test"

img_rows, img_cols = 224,224
batch_size = 32

# ==========================================
# DATA AUGMENTATION
# ==========================================

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.25,
    shear_range=0.15,
    brightness_range=[0.8,1.2],
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# ==========================================
# LOAD DATA
# ==========================================

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_rows,img_cols),
    batch_size=batch_size,
    class_mode='categorical',
    seed=42
)

validation_generator = val_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(img_rows,img_cols),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_rows,img_cols),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# ==========================================
# CHECK DATA
# ==========================================

nb_train_samples = train_generator.samples
nb_validation_samples = validation_generator.samples
nb_test_samples = test_generator.samples

print("Train samples:", nb_train_samples)
print("Validation samples:", nb_validation_samples)
print("Test samples:", nb_test_samples)

print("Class indices:", train_generator.class_indices)

num_classes = len(train_generator.class_indices)
print("Model set to train", num_classes, "classes")

if nb_train_samples>0 and nb_validation_samples>0 and nb_test_samples>0:
    print("Generators are set!")
    print("Dataset ready for training.")

Found 5600 images belonging to 8 classes.
Found 1200 images belonging to 8 classes.
Found 1200 images belonging to 8 classes.
Train samples: 5600
Validation samples: 1200
Test samples: 1200
Class indices: {'dyed-lifted-polyps': 0, 'dyed-resection-margins': 1, 'esophagitis': 2, 'normal-cecum': 3, 'normal-pylorus': 4, 'normal-z-line': 5, 'polyps': 6, 'ulcerative-colitis': 7}
Model set to train 8 classes
Generators are set!
Dataset ready for training.


# PREPARE MODELS

In [14]:
# Model A
#EfficientNetB0
builder_a = DCNN_A + '_builder'

#TRANSFER LEARNING
def builder_a(model_input):
    builder_a = trainable_model_a(weights='imagenet', 
                                    include_top=False, 
                                    input_tensor = model_input)

#PARTIAL LAYER FREEZING
    for layer in builder_a.layers:
        layer.trainable = False
        
    for layer in builder_a.layers:
        layer._name = layer.name + '_' + DCNN_A
        
    for BatchNormalization in builder_a.layers:
        BatchNormalization.trainable = False
    
#LAYER COMPRESSION
    x = builder_a.layers[-17].output #Equivalent to one (1) CORE block deduction.
    
#AUXILIARY FUSING LAYER (AuxFL)
    x = Conv2D(192, 1, padding='valid', activation='selu', kernel_initializer='lecun_normal')(x)
    x = AveragePooling2D(1, 1)(x)
    x = AlphaDropout(0.2)(x)

    dcnn_a = Model(inputs=builder_a.input, outputs=x, name=DCNN_A)
    return dcnn_a

#INITIALIZE THE MODEL
dcnn_a = builder_a(model_input)

#PLOT THE MODEL STRUCTURE
print("PLEASE CHECK THE ENTIRE MODEL UP TO THE END")
dcnn_a.summary()
print("successfully built!")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
PLEASE CHECK THE ENTIRE MODEL UP TO THE END


Model: "DCNN_A"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 2,949,427 (11.25 MB)

 Trainable params: 37,056 (144.75 KB)

 Non-trainable params: 2,912,371 (11.11 MB)

successfully built!


In [15]:
# Model B
#MobileNetV2
builder_b = DCNN_B + '_builder'

#TRANSFER LEARNING
def builder_b(model_input):
    builder_b = trainable_model_b(weights='imagenet', 
                                    include_top=False, 
                                    input_tensor = model_input)

#PARTIAL LAYER FREEZING
    for layer in builder_b.layers:
        layer.trainable = False
        
    for layer in builder_b.layers:
        layer._name = layer.name + '_' + DCNN_B
        
    for BatchNormalization in builder_b.layers:
        BatchNormalization.trainable = False
    
#LAYER COMPRESSION
    x = builder_b.layers[-39].output #Equivalent to four (4) CORE block deduction.
    
#AUXILIARY FUSING LAYER (AuxFL)
    x = Conv2D(192, 8, padding='valid', activation='selu', kernel_initializer='lecun_normal')(x)
    x = AveragePooling2D(1, 1)(x)
    x = AlphaDropout(0.2)(x)

    dcnn_b = Model(inputs=builder_b.input, outputs=x, name=DCNN_B)
    return dcnn_b

#INITIALIZE THE MODEL
dcnn_b = builder_b(model_input)

#PLOT THE MODEL STRUCTURE
print("PLEASE CHECK THE ENTIRE MODEL UP TO THE END")
dcnn_b.summary()
print(" successfully built!")

/tmp/ipykernel_55/216559017.py:7: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  builder_b = trainable_model_b(weights='imagenet',


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
PLEASE CHECK THE ENTIRE MODEL UP TO THE END


Model: "DCNN_B"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 1,738,496 (6.63 MB)

 Trainable params: 1,179,840 (4.50 MB)

 Non-trainable params: 558,656 (2.13 MB)

 successfully built!


In [16]:
# Model C
#ResNet50V2
builder_c = DCNN_C + '_builder'

#TRANSFER LEARNING
def builder_c(model_input):
    builder_c = trainable_model_c(weights='imagenet', 
                                    include_top=False, 
                                    input_tensor = model_input)

#PARTIAL LAYER FREEZING
    for layer in builder_c.layers:
        layer.trainable = False
        
    for layer in builder_c.layers:
        layer._name = layer.name + '_' + DCNN_C
        
    for BatchNormalization in builder_c.layers:
        BatchNormalization.trainable = False
    
#LAYER COMPRESSION
    x = builder_c.layers[-117].output #Equivalent to two (2) CORE block deduction.
    
 #AUXILIARY FUSING LAYER (AuxFL)   
    x = Conv2D(192, 6, padding='valid', activation='selu', kernel_initializer='lecun_normal')(x)
    x = AveragePooling2D(3, 3)(x)
    x = AlphaDropout(0.2)(x)

    dcnn_c = Model(inputs=builder_c.input, outputs=x, name=DCNN_C)
    return dcnn_c

#INITIALIZE THE MODEL
dcnn_c = builder_c(model_input)

#PLOT THE MODEL STRUCTURE
print("PLEASE CHECK THE ENTIRE MODEL UP TO THE END")
dcnn_c.summary()
print("successfully built!")

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
PLEASE CHECK THE ENTIRE MODEL UP TO THE END


Model: "DCNN_C"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_conv[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_preac… │ (None, 56, 56,    │        256 │ pool1_pool[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_preac… │ (None, 56, 56,    │          0 │ conv2_block1_pre… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,096 │ conv2_block1_pre… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_pad  │ (None, 58, 58,    │          0 │ conv2_block1_1_r… │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_2_p… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_pre… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_out    │ (None, 56, 56,    │          0 │ conv2_block1_0_c

 Total params: 4,710,592 (17.97 MB)

 Trainable params: 3,539,136 (13.50 MB)

 Non-trainable params: 1,171,456 (4.47 MB)

successfully built!


# PREPARE FUSION

In [17]:
#RE-INITIALIZE FOR FUSION
dcnn_a = builder_a(model_input)


print("Accomplished Pre-training and ready for fusion")

Accomplished Pre-training and ready for fusion


In [18]:
#FUSE THE MODELS INTO A SINGLE PIPELINE
models = [dcnn_a, 
          dcnn_b,
          dcnn_c]

print("Fusion success!")
print("Ready to connect with its ending layers!")

Fusion success!
Ready to connect with its ending layers!


In [19]:
# ==========================================
# REQUIRED IMPORTS FOR ATTENTION BLOCK
# ==========================================

import tensorflow as tf
from tensorflow.keras.layers import Activation, Lambda
from tensorflow.keras import backend as K

In [20]:
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    Dense,
    Add,
    Multiply,
    GlobalAveragePooling2D,
    BatchNormalization,
    Activation,
    Lambda,
    AlphaDropout,
    Reshape
)

from tensorflow.keras import backend as K

# Fusion Residual Block (FuRB)

In [21]:
# ==========================================
# Build Fusion Residual Block (FuRB)
# ==========================================

def hafusionnet_builder(models, model_input):

    outputs = [m.output for m in models]

    # INITIAL FUSION LAYER
    y = Add(name='InitialFusionLayer')(outputs)
    y = feature_attention_block(y)

    # FuRB LAYER
    y_bn1 = BatchNormalization()(y)
    y_selu1 = tf.keras.activations.selu(y_bn1)

    y_conv1 = Conv2D(192, 1, kernel_initializer='lecun_normal')(y_selu1)

    y_bn2 = BatchNormalization()(y_conv1)
    y_selu2 = tf.keras.activations.selu(y_bn2)

    y_conv2 = Conv2D(192, 1, kernel_initializer='lecun_normal')(y_selu2)

    # Residual merge
    y_merge = Add(name='FuRB')([y, y_conv2])

    # FINE-TUNING LAYER
    y = GlobalAveragePooling2D()(y_merge)
    y = AlphaDropout(0.5)(y)

    prediction = Dense(num_classes,
                       activation='softmax',
                       name='Softmax_Classifier' + architecture)(y)

    model = Model(model_input, prediction, name=architecture)

    return model


# ==========================================
# Instantiate Model
# ==========================================

hafusionnet = hafusionnet_builder(models, model_input)
hafusionnet._name = "HAFusionNet"
model_name = hafusionnet._name

print()
print("PLEASE CHECK THE MODEL UP TO THE END")
print()

hafusionnet.summary()

print("The", model_name, "is now complete and ready for compilation and training!")


PLEASE CHECK THE MODEL UP TO THE END



Model: "HAFusionNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 9,526,839 (36.34 MB)

 Trainable params: 4,883,588 (18.63 MB)

 Non-trainable params: 4,643,251 (17.71 MB)

The HAFusionNet is now complete and ready for compilation and training!


In [22]:
# Check the layer, the final layer should be FuRB!
last_conv_layer_name = hafusionnet.layers[-4].name

if last_conv_layer_name == 'FuRB':
    print("CORRECT LAYER SELECTED:", last_conv_layer_name)
else:
    print("INCORRECT LAYER SELECTED:", last_conv_layer_name)
    print("Please Reselect")

CORRECT LAYER SELECTED: FuRB


In [23]:
for model in models:
    for layer in model.layers[:-30]:
        layer.trainable = False
    for layer in model.layers[-30:]:
        layer.trainable = True

# SAVE POINT

In [24]:
#save json file
model_dir = 'model/'

if not os.path.exists(model_dir):
    os.mkdir(model_dir)
    print('Model directory', model_dir, 'successfully created')
else:
    print('Model directory already exist, no new directory made.')

print()
print('-'*50)
print('Model directory is available for saving the', model_name, 'model!')
print('-'*50)

Model directory model/ successfully created

--------------------------------------------------
Model directory is available for saving the HAFusionNet model!
--------------------------------------------------


# COMPILE MODEL

In [25]:
# MODEL COMPILATION WITH HYPER-PARAMETERS, LOSS FUNCTIONS AND TRAINING!
hafusionnet.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy']) 

reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=2,
                              verbose=1, mode='max')

callbacks = [reduce_lr]

print('-'*50)
print('Successfully compiled the', model_name, 'model!')
print('You may now proceed in training the', model_name, 'model!')
print('-'*50)

--------------------------------------------------
Successfully compiled the HAFusionNet model!
You may now proceed in training the HAFusionNet model!
--------------------------------------------------


# TRAIN THE Multi-Fused CNN MODEL (HAFusionNet)

In [ ]:
# MODEL TRAINING

# Set training time
start_time = time.time()
print('*'*50)
print("Training", model_name)
print('*'*50)
print('-'*50)
print("Training time is being measured")
print('-'*50)

history = hafusionnet.fit(train_generator, 
                          steps_per_epoch=nb_train_samples // batch_size,
                          epochs=epochs, 
                          validation_data=validation_generator,
                          callbacks=callbacks, 
                          validation_steps=nb_validation_samples // batch_size, 
                          verbose=1)

print()
print("MODEL SERIALIZING WAIT FOR A MOMENT...")

elapsed_time = time.time() - start_time
train_time = time.strftime("%H:%M:%S", time.gmtime(elapsed_time))

print()
print()
print(train_time, 'train_time')
print()
print(elapsed_time, 'Seconds')
print()
print()
print("MODEL SERIALIZATION DONE!")

**************************************************
Training HAFusionNet
**************************************************
--------------------------------------------------
Training time is being measured
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/2


I0000 00:00:1772957881.041684     129 service.cc:152] XLA service 0x78f4480b7b30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772957881.041724     129 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1772957881.041731     129 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1772957886.109592     129 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-08 08:18:15.165006: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-08 08:18:15.309713: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-08 08:18:15.654247: E external/local_xl

152/175 ━━━━━━━━━━━━━━━━━━━━ 13s 598ms/step - accuracy: 0.2516 - loss: 2.4137

# SAVE THE TRAINED MODEL

In [ ]:
import os

# Create directory if it doesn't exist
save_path = 'model/' + architecture
os.makedirs(save_path, exist_ok=True)

# SAVE MODEL WEIGHTS (correct filename format)
weights_path = save_path + '/' + architecture + '.weights.h5'
hafusionnet.save_weights(weights_path)

# SAVE TRAINING HISTORY
save_h('model/', history.history)

# Prompt
print("-"*50)
print()
print("The model weights and history of", model_name, "have been successfully trained and saved!")
print()
print("Weights saved at:", weights_path)
print()
print("-"*50)

# EVALUATION WITH VALIDATION DATASET

In [ ]:
# build the model using the builder function
model = hafusionnet_builder(models, model_input)

# correct weights path
weights_path = 'model/' + architecture + '/' + architecture + '.weights.h5'

# load weights
model.load_weights(weights_path)

print("The", model_name, "is successfully loaded!")

In [ ]:
# ==========================================
# PREDICTIONS
# ==========================================
y_pred = model.predict(validation_generator, steps=len(validation_generator))

In [ ]:
print(history.history.keys())

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Accuracy Curve
plt.figure(figsize=(6,4))
plt.plot(epochs_range, history.history['accuracy'], label='Train Accuracy')
plt.plot(epochs_range, history.history['val_accuracy'], label='Validation Accuracy')

plt.title('Training and Validation Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


# Loss Curve
plt.figure(figsize=(6,4))
plt.plot(epochs_range, history.history['loss'], label='Train Loss')
plt.plot(epochs_range, history.history['val_loss'], label='Validation Loss')

plt.title('Training and Validation Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Modify only as needed

# Figure
dpi = 1000
plt.rcParams.update({'figure.dpi': dpi})
figsize = (12, 12)

# Markers
marker_train_accuracy = 's'
marker_validation_accuracy = 'x'
marker_train_loss = 'o'
marker_validation_loss = '|'
marker_fillstyle_train = 'none'
marker_fillstyle_validation = 'none'
marker_plot_markersize = 25
marker_plot_markerwidth = 3

# Lines
line_style_train = '-' 
line_style_validation = '--'
line_width_train = 5
line_width_val = line_width_train
line_color_train_accuracy = 'black'
line_color_val_accuracy = 'black'
line_color_train_loss = 'black'
line_color_val_loss = 'black'

# Labels
train_accuracy_label = 'Train Acc'
validation_accuracy_label = 'Val Acc'
train_loss_label = 'Train Loss'
validation_loss_label = 'Val Loss'

x_label_font_size = 22
y_label_font_size = x_label_font_size

# Use font supported in Kaggle / Matplotlib
x_label_font = 'DejaVu Sans'
y_label_font = x_label_font

# Ticks
spine_axis_thickness = 4
tick_font_size = 22
tick_length = 12
tick_width = spine_axis_thickness

# Legend
legend_border_pad = 0.35
legend_line_width = 5
legend_font_size = 50
legend_edge_color = 'black'
legend_label_spacing = 0.5
legend_location = 'best'
legend_ncol = 1
legend_font = 'DejaVu Sans'
legend_has_frame = True

In [ ]:
#Convergence
plt.style.use("default")
plt.figure(figsize = figsize, 
           dpi = dpi, 
           edgecolor = 'black', 
           facecolor = 'white', 
           linewidth = 0)

plt.tight_layout()
plt.rc('xtick', labelsize = tick_font_size, direction="in") 
plt.rc('ytick', labelsize = tick_font_size, direction="in") 

fig, ax = plt.subplots(figsize = figsize)
plt.gcf().subplots_adjust(bottom = 0.15)
plt.setp(ax.spines.values(), linewidth = spine_axis_thickness)

plt.tick_params(length = tick_length, 
                width = tick_width, 
                right = True, 
                top = True)

plt.plot(np.arange(1, epochs + 1), 
         history.history["accuracy"], 
         mew = marker_plot_markerwidth, 
         color = line_color_train_accuracy, 
         lw = line_width_train, 
         marker = marker_train_accuracy, 
         markersize = marker_plot_markersize, 
         fillstyle = marker_fillstyle_train, 
         ls = line_style_train, 
         label = train_accuracy_label)

plt.plot(np.arange(1, epochs + 1), 
         history.history["val_accuracy"], 
         mew = marker_plot_markerwidth, 
         color = line_color_val_accuracy, 
         lw = line_width_val, 
         marker = marker_validation_accuracy, 
         markersize = marker_plot_markersize, 
         fillstyle = marker_fillstyle_validation, 
         ls = line_style_validation,  
         label = validation_accuracy_label)

plt.plot(np.arange(1, epochs + 1), 
         history.history["loss"], 
         mew = marker_plot_markerwidth, 
         color = line_color_train_loss, 
         lw = line_width_train, 
         marker = marker_train_loss, 
         markersize = marker_plot_markersize, 
         fillstyle = marker_fillstyle_train, 
         ls = line_style_train, label = train_loss_label)

plt.plot(np.arange(1, epochs + 1), 
         history.history["val_loss"], 
         mew = marker_plot_markerwidth, 
         color = line_color_val_loss, 
         lw = line_width_val, 
         marker = marker_validation_loss, 
         markersize = marker_plot_markersize, 
         fillstyle = marker_fillstyle_validation, 
         ls = line_style_validation,  
         label = validation_loss_label)

plt.xlabel("Epochs", fontfamily = x_label_font, fontsize = x_label_font_size, color ='black')
plt.ylabel("Accuracy/Loss", fontfamily = y_label_font, fontsize = y_label_font_size, color = 'black')

legend = plt.legend(loc = legend_location, 
                    ncol = legend_ncol, 
                    frameon = legend_has_frame, 
                    fontsize=legend_font_size, 
                    edgecolor=legend_edge_color, 
                    borderpad=legend_border_pad, 
                    labelspacing=legend_label_spacing)

frame = legend.get_frame()
legend.get_frame().set_linewidth(legend_line_width)
legend.get_frame().set_edgecolor(legend_edge_color)
plt.setp(legend.texts, family = legend_font)
plt.tight_layout()

# EVALUATION WITH TEST DATASET

In [ ]:
# Confusion Matrix (Normalized)
fontsize = 12

def confusion_matrix_test(cm, classes,
                          normalize=True,
                          title=model.name + ' Val',
                          cmap=plt.cm.binary):

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, fontsize=18)
    plt.colorbar(orientation='vertical')

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=0, fontsize=fontsize)
    plt.yticks(tick_marks, classes, rotation=0, fontsize=fontsize)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print("Confusion matrix, without normalization")

    print(cm)

    thresh = cm.max() / 2.

    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, '{:.2f}%'.format(cm[i, j]*100),
                 fontsize=9, weight='bold',
                 horizontalalignment="center",
                 verticalalignment='center',
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label', fontsize=18)
    plt.xlabel('Predicted label', fontsize=18)

In [ ]:
# Get true labels from validation generator
Y_test = validation_generator.classes

# Predict probabilities
y_pred = model.predict(validation_generator)

print("Y_test shape:", Y_test.shape)
print("Predictions shape:", y_pred.shape)

# Test_Generator.Classes

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

n_classes = 8

# True labels
test_labels = test_generator.classes

# Convert labels to binary
Y_test_bin = label_binarize(test_labels, classes=[0,1,2,3,4,5,6,7])

# Model predictions (probabilities)
y_pred_test = model.predict(test_generator)

fpr = dict()
tpr = dict()
roc_auc = dict()

# Compute ROC for each class
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(Y_test_bin[:, i], y_pred_test[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot
plt.figure(figsize=(7,7), dpi=150)

for i in range(n_classes):
    plt.plot(
        fpr[i],
        tpr[i],
        lw=2,
        label='Class {} (AUC = {:.3f})'.format(i, roc_auc[i])
    )

# Diagonal reference
plt.plot([0,1],[0,1],'k--',lw=2)

# Labels (same style as your first code)
plt.title(model.name + ' Test ROC Curve', fontsize=18, fontfamily='DejaVu Sans')
plt.xlabel('Specificity', fontsize=18, fontfamily='DejaVu Sans')
plt.ylabel('Sensitivity', fontsize=18, fontfamily='DejaVu Sans')

# Tick formatting
plt.tick_params(length=5,
                width=2,
                right=True,
                top=True,
                labelsize=12)

plt.rc('xtick', direction="in")
plt.rc('ytick', direction="in")

legend = plt.legend(loc="lower right", fontsize=15, labelspacing=1)
plt.setp(legend.texts, family='DejaVu Sans')

plt.tight_layout()
plt.show()

In [ ]:
#Sanity check for validation
#model.evaluate(validation_generator, return_dict=True)
# Compile the model first
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Sanity check for validation
results = model.evaluate(validation_generator, return_dict=True)

print(results)

# EVALUATION WITH TEST DATASET

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, mean_squared_error, mean_squared_log_error

# Predict test data
y_pred_test = model.predict(
    test_generator,
    steps=int(np.ceil(test_generator.samples / batch_size))
)

# True labels
test_labels = test_generator.classes

# Accuracy
accuracy_test = accuracy_score(test_labels, y_pred_test.argmax(axis=1))
print('The test accuracy of the ' + model.name + ' is:', accuracy_test)

# Mean Squared Error
mse_test = mean_squared_error(test_labels, y_pred_test.argmax(axis=1))
print('The test MSE of the ' + model.name + ' is:', mse_test)

# Mean Squared Log Error
msle_test = mean_squared_log_error(test_labels, y_pred_test.argmax(axis=1))
print('The test MSLE of the ' + model.name + ' is:', msle_test)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

plt.style.use("default")

# Class names (Kvasir v2)
target_names = [
    'dyed-lifted-polyps',
    'dyed-resection-margins',
    'esophagitis',
    'normal-cecum',
    'normal-pylorus',
    'normal-z-line',
    'polyps',
    'ulcerative-colitis'
]

# Classification report
print(classification_report(
    test_labels,
    y_pred_test.argmax(axis=1),
    target_names=target_names,
    digits=4
))

# Compute confusion matrix
cm = confusion_matrix(test_labels, y_pred_test.argmax(axis=1))

# Plot confusion matrix
plt.figure(figsize=(7,6), dpi=150)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=target_names,
    yticklabels=target_names,
    linewidths=0.5,
    linecolor='gray',
    cbar=True
)

plt.title("Confusion Matrix", fontsize=12)
plt.xlabel("Predicted Labels", fontsize=11)
plt.ylabel("True Labels", fontsize=11)

plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

# Binarize labels for multi-class PR curve
n_classes = len(np.unique(test_labels))
y_test_bin = label_binarize(test_labels, classes=np.arange(n_classes))

plt.figure(figsize=(6,5), dpi=150)

for i in range(n_classes):
    precision, recall, _ = precision_recall_curve(
        y_test_bin[:, i],
        y_pred_test[:, i]
    )

    ap = average_precision_score(y_test_bin[:, i], y_pred_test[:, i])

    plt.plot(recall, precision, lw=2,
             label=f'Class {i} (AP = {ap:.2f})')

plt.title(model.name + ' Test P-R Curve', fontsize=16)
plt.xlabel('Recall', fontsize=14)
plt.ylabel('Precision', fontsize=14)

plt.tick_params(length=5, width=2, right=True, top=True, labelsize=11)

plt.legend(loc="lower left", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
#Sanity check for test
model.evaluate(test_generator, return_dict=True)

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2


def get_flops(model):

    concrete = tf.function(lambda x: model(x))
    concrete_func = concrete.get_concrete_function(
        tf.TensorSpec([1] + list(model.input.shape[1:]), model.input.dtype)
    )

    frozen_func = convert_variables_to_constants_v2(concrete_func)
    graph_def = frozen_func.graph.as_graph_def()

    with tf.Graph().as_default() as graph:
        tf.graph_util.import_graph_def(graph_def, name="")

        run_meta = tf.compat.v1.RunMetadata()
        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        flops = tf.compat.v1.profiler.profile(
            graph=graph,
            run_meta=run_meta,
            cmd="op",
            options=opts
        )

    return flops.total_float_ops


def cost_compute():

    flops = get_flops(model) / 1e9
    params = model.count_params() / 1e6
    trainable_params = sum([K.count_params(w) for w in model.trainable_weights]) / 1e6

    print("FLOPs: {:.2f} GFLOPs".format(flops))
    print("Params: {:.2f} M".format(params))
    print("Trainable Params: {:.2f} M".format(trainable_params))


cost_compute()